# 6. Persistence / Checkpointers (Standalone)

A checkpointer is a property of the *graph*, not something special about agents
— any `StateGraph` gets persistent, resumable state across separate `.invoke()`
calls just by compiling it with `checkpointer=...` and reusing the same
`thread_id`. Demonstrated with a trivial counter: no LLM, no tools at all.

**Prerequisites:** none — this notebook makes no model calls.

### Setup

This cell makes the project's shared `tools`/`models` packages importable
regardless of where Jupyter's working directory actually is (it's usually
this notebook's own folder, not the repo root), and loads `.env` plus any
cached secrets in `.env.local` (populated by `scripts/lib/env.sh` the first
time you've run `scripts/start_app.sh` / `scripts/start_infra.sh`).

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
load_dotenv(project_root / ".env.local", override=True)  # cached secrets, if resolve_env() has run at least once
print("Project root on sys.path:", project_root)

In [ ]:
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph


class CounterState(TypedDict):
    count: int
    history: list[str]


def increment(state: CounterState) -> dict:
    # .get() with defaults: on a brand-new thread there's no checkpointed
    # state yet, so these keys won't exist in the incoming state at all.
    count = state.get("count", 0) + 1
    history = state.get("history", []) + [f"incremented to {count}"]
    return {"count": count, "history": history}


graph = StateGraph(CounterState)
graph.add_node("increment", increment)
graph.add_edge(START, "increment")
graph.add_edge("increment", END)
compiled = graph.compile(checkpointer=InMemorySaver())

## Call it three times on the same thread — count accumulates

In [ ]:
thread = {"configurable": {"thread_id": "notebook-demo"}}

# Deliberately invoked with an EMPTY dict, not {"count": 0, ...} — passing an
# explicit count would overwrite the checkpoint back to zero every call
# instead of accumulating.
for _ in range(3):
    result = compiled.invoke({}, thread)
    print(result)

## A different `thread_id` starts fresh

In [ ]:
other_thread = {"configurable": {"thread_id": "a-different-session"}}
print(compiled.invoke({}, other_thread))

## 🧪 Playground

**1. Confirm isolation** — call `"notebook-demo"` again and confirm it continues from 3, unaffected by the other thread.

In [ ]:
# TODO: compiled.invoke({}, thread) again


**2. Inspect checkpoint history** — `list(compiled.get_state_history(thread))` shows every past checkpoint for a thread, oldest last.

In [ ]:
# TODO: list(compiled.get_state_history(thread)) and print each snapshot's .values


**3. A second field** — add a `last_incremented_by: str` field to `CounterState` and have `increment` accept a name parameter somehow (hint: you'll need to pass it via the invoke input on a fresh thread, since input on a checkpointed thread only patches, it doesn't reset).

In [ ]:
# TODO: extend CounterState and increment
